In [121]:
from pathlib import Path
import pandas as pd
import zipfile
from collections import Counter
import matplotlib.pyplot as plt
import csv

#### Time Period wise view

Exploration DataFrames

In [122]:
labelled_channels = pd.read_csv("/Volumes/StudyNProjects/Python/ESA-Mission1/labels.csv")
anamoly_types = pd.read_csv("ESA-Mission1/anomaly_types.csv")
channels = pd.read_csv("ESA-Mission1/channels.csv")
telecommand = pd.read_csv("/Volumes/StudyNProjects/Python/ESA-Mission1/telecommands.csv")
telecommand_folder = Path("/Volumes/StudyNProjects/Python/ESA-Mission1/telecommands")
unzip_telecommand = list(telecommand_folder.glob("*.zip"))

In [123]:
labelled_df = pd.DataFrame(labelled_channels)
#
labelled_df["StartTime"] = pd.to_datetime(labelled_df["StartTime"])
labelled_df["EndTime"] = pd.to_datetime(labelled_df["EndTime"])
labelled_df = labelled_df.set_index("StartTime").sort_index()
#labelled_df["Sampled_startTime"] = labelled_df["StartTime"].dt.strftime("%d-%b-%Y %H:%M")
#labelled_df["Sampled_endTime"] = labelled_df["EndTime"].dt.strftime("%d-%b-%Y %H:%M")
print(labelled_df)
labelled_df.index = pd.to_datetime(labelled_df.index)

                                      ID     Channel  \
StartTime                                              
2000-02-10 01:31:16.347000+00:00  id_151  channel_44   
2000-02-10 01:31:46.347000+00:00  id_151  channel_41   
2000-02-10 01:31:46.347000+00:00  id_151  channel_43   
2000-02-10 01:31:46.347000+00:00  id_151  channel_45   
2000-02-10 01:31:46.347000+00:00  id_151  channel_42   
...                                  ...         ...   
2013-12-04 16:32:59.943000+00:00  id_189  channel_14   
2013-12-04 16:40:29.943000+00:00  id_189  channel_14   
2013-12-04 17:41:59.943000+00:00  id_189  channel_14   
2013-12-04 17:56:59.943000+00:00  id_189  channel_29   
2013-12-04 18:22:29.943000+00:00  id_189  channel_21   

                                                          EndTime  
StartTime                                                          
2000-02-10 01:31:16.347000+00:00 2000-02-10 01:35:16.347000+00:00  
2000-02-10 01:31:46.347000+00:00 2000-02-10 01:35:16.347000+00:00  

In [124]:
monthly_time_period = labelled_df.loc["2001-11-01 10:30":"2001-12-01 16:15"]
print(monthly_time_period)
monthly_time_period.nunique()

                                    ID     Channel  \
StartTime                                            
2001-11-24 17:31:45.561000+00:00  id_3  channel_14   
2001-11-24 17:31:45.561000+00:00  id_3  channel_29   
2001-11-24 17:31:45.561000+00:00  id_3  channel_21   

                                                          EndTime  
StartTime                                                          
2001-11-24 17:31:45.561000+00:00 2001-11-25 00:16:45.561000+00:00  
2001-11-24 17:31:45.561000+00:00 2001-11-24 22:36:15.561000+00:00  
2001-11-24 17:31:45.561000+00:00 2001-11-24 22:10:45.561000+00:00  


ID         1
Channel    3
EndTime    3
dtype: int64

During this month/week/day, what were the channels doing?
Were there labelled events?
Were there telecommands?
Which subsystems were active?
Which channels changed?

In [125]:
no_of_channel_TP = monthly_time_period["Channel"].count()

#subSystem_TP = monthly_time_period["Channel"].isin(channels["Channel"])
subSystem_TP = channels.loc[channels["Channel"].isin(monthly_time_period["Channel"])]
anamoly_type_TP = anamoly_types.loc[anamoly_types["ID"].isin(monthly_time_period["ID"])]
print(anamoly_type_TP)
#print("SubSystems Active on the time period\n", subSystem_TP["Subsystem"])
print("Number of Channel Labelled event within Time Period\n",no_of_channel_TP)
print("Labels inside the window\n", monthly_time_period)

     ID    Class    Subclass Category Dimensionality Locality       Length
2  id_3  class_7  subclass_1  Anomaly   Multivariate   Global  Subsequence
Number of Channel Labelled event within Time Period
 3
Labels inside the window
                                     ID     Channel  \
StartTime                                            
2001-11-24 17:31:45.561000+00:00  id_3  channel_14   
2001-11-24 17:31:45.561000+00:00  id_3  channel_29   
2001-11-24 17:31:45.561000+00:00  id_3  channel_21   

                                                          EndTime  
StartTime                                                          
2001-11-24 17:31:45.561000+00:00 2001-11-25 00:16:45.561000+00:00  
2001-11-24 17:31:45.561000+00:00 2001-11-24 22:36:15.561000+00:00  
2001-11-24 17:31:45.561000+00:00 2001-11-24 22:10:45.561000+00:00  


In [149]:
start_time = "2001-11-01 10:30"
end_time = "2001-12-01 16:15"
data_frame =[]
telecommand_2 = Path("/Volumes/StudyNProjects/Python/ESA-Mission1/telecommands/telecommand_2.zip")


for zips in unzip_telecommand:
    with zipfile.ZipFile(zips, "r") as zip:
       for interal_file in zip.namelist():
           with zip.open(interal_file) as fileStream:
               df= pd.read_pickle(fileStream)
               mask = df.index.month.isin([11,12]) & (df.index.year ==2001)
               
               selected_telecommand = df.loc[mask]

               if not selected_telecommand.empty:
                   telecommand_name = selected_telecommand.columns[0]

                   temp = pd.DataFrame({
                       "date_time" : selected_telecommand.index,
                        "telecommand":telecommand_name,
                        "value": selected_telecommand[telecommand_name].values
                        })
                   data_frame.append(temp)
                   
                
final_dataFrame = pd.concat(data_frame, ignore_index=True)
print(final_dataFrame.count())
un_value = final_dataFrame["telecommand"].value_counts()
print(un_value)
'''
with zipfile.ZipFile(telecommand_2, "r") as files:
    for inside_file in files.namelist():
        # 2. FIXED: Changed 'zip.open' to 'files.open' to match your variable above
        with files.open(inside_file) as filess:
            df = pd.read_pickle(filess)
            #day_mask = ((df.index.day >=1) &(df.index.day<=31)) & ((df.index.month>=11)&(df.index.month<=12))
            mask = df.index.month.isin([11,12]) & (df.index.year ==2001)
            if mask.any():
                data_frame.append(df) 

# Combine all the dataframes into one master DataFrame
#result = pd.concat(data_frame, ignore_index=False)

# Print the final combined DataFrame (instead of the raw list of dataframes)
print(data_frame)

'''


date_time      6259
telecommand    6259
value          6259
dtype: int64
telecommand
telecommand_270    2554
telecommand_493    1024
telecommand_198     643
telecommand_233     221
telecommand_232     180
                   ... 
telecommand_587       2
telecommand_41        1
telecommand_276       1
telecommand_441       1
telecommand_372       1
Name: count, Length: 66, dtype: int64


'\nwith zipfile.ZipFile(telecommand_2, "r") as files:\n    for inside_file in files.namelist():\n        # 2. FIXED: Changed \'zip.open\' to \'files.open\' to match your variable above\n        with files.open(inside_file) as filess:\n            df = pd.read_pickle(filess)\n            #day_mask = ((df.index.day >=1) &(df.index.day<=31)) & ((df.index.month>=11)&(df.index.month<=12))\n            mask = df.index.month.isin([11,12]) & (df.index.year ==2001)\n            if mask.any():\n                data_frame.append(df) \n\n# Combine all the dataframes into one master DataFrame\n#result = pd.concat(data_frame, ignore_index=False)\n\n# Print the final combined DataFrame (instead of the raw list of dataframes)\nprint(data_frame)\n\n'

### Period Summary  


#### Labelled Events Between the periods

In [127]:
#print(labelled_channels)
start_time = "2001-11-01 10:30"
end_time = "2001-12-01 16:15"
labelled_channels['StartTime'] = pd.to_datetime(labelled_channels['StartTime'])
labelled_channels['EndTime'] = pd.to_datetime(labelled_channels['EndTime'])
mask_s = labelled_channels["StartTime"].dt.month.isin([11,12]) & (labelled_channels["StartTime"].dt.year == 2001)
mask_e = labelled_channels["EndTime"].dt.month.isin([11,12]) & (labelled_channels["EndTime"].dt.year == 2001)
combines = mask_s & mask_e
labelled_event_bt_period = labelled_channels[combines]
print(labelled_event_bt_period.count())



ID           6
Channel      6
StartTime    6
EndTime      6
dtype: int64


#### Number of Affected Channels

In [128]:
affected_channels = labelled_event_bt_period["ID"].isin(anamoly_types["ID"])
temp = labelled_event_bt_period[affected_channels]
afftected_ones = anamoly_types["ID"].isin(temp["ID"])
print(anamoly_types[afftected_ones])


     ID    Class    Subclass Category Dimensionality Locality       Length
2  id_3  class_7  subclass_1  Anomaly   Multivariate   Global  Subsequence
5  id_6  class_7  subclass_1  Anomaly   Multivariate    Local  Subsequence


#### Number of Subsystem Involved

In [129]:
channels_in_lbl_events = labelled_event_bt_period["Channel"]
#print(channels_in_lbl_events)
compare = channels["Channel"].isin(channels_in_lbl_events)
subSystem_involved = channels[compare]
print(subSystem_involved["Subsystem"].unique())

['subsystem_6']


In [130]:
'''
data_frame = []

for zips in unzip_telecommand:
    with zipfile.ZipFile(zips, "r") as zip:
        for internal_file in zip.namelist():
            with zip.open(internal_file) as fileStream:
                df = pd.read_pickle(fileStream)

                mask = (df.index >= start_time) & (df.index <= end_time)
                selected_telecommand = df.loc[mask]

                if not selected_telecommand.empty:
                    telecommand_name = selected_telecommand.columns[0]

                    temp = pd.DataFrame({
                        "date_time": selected_telecommand.index,
                        "telecommand": telecommand_name,
                        "value": selected_telecommand[telecommand_name].values
                    })

                    data_frame.append(temp)

if data_frame:
    final_dataFrame = pd.concat(data_frame, ignore_index=True)
    final_dataFrame = final_dataFrame.sort_values("date_time")

    number_of_telecommand_occurrences = len(final_dataFrame)
    number_of_unique_telecommands = final_dataFrame["telecommand"].nunique()

    print("Number of telecommand occurrences:", number_of_telecommand_occurrences)
    print("Number of unique telecommands:", number_of_unique_telecommands)
    print(final_dataFrame["telecommand"].value_counts().head(10))
else:
    final_dataFrame = pd.DataFrame(columns=["date_time", "telecommand", "value"])

    print("Number of telecommand occurrences: 0")
    print("Number of unique telecommands: 0")

    start_time = pd.to_datetime("2001-11-01 10:30")
end_time = pd.to_datetime("2001-12-01 16:15")

# -------------------------------
# Labels overlapping selected period
# -------------------------------
labelled_channels["StartTime"] = pd.to_datetime(labelled_channels["StartTime"])
labelled_channels["EndTime"] = pd.to_datetime(labelled_channels["EndTime"])

label_mask = (
    (labelled_channels["StartTime"] <= end_time) &
    (labelled_channels["EndTime"] >= start_time)
)

period_labels = labelled_channels[label_mask]

number_of_labelled_events = period_labels["ID"].nunique()
number_of_label_rows = len(period_labels)
number_of_affected_channels = period_labels["Channel"].nunique()

# -------------------------------
# Event type mapping
# -------------------------------
period_labels_with_types = period_labels.merge(anamoly_types, on="ID", how="left")

# -------------------------------
# Subsystems involved
# -------------------------------
affected_channel_names = period_labels["Channel"].unique()

period_channel_metadata = channels[
    channels["Channel"].isin(affected_channel_names)
]

number_of_subsystems = period_channel_metadata["Subsystem"].nunique()

# -------------------------------
# Period summary
# -------------------------------
print("Selected period:", start_time, "to", end_time)
print("Number of labelled events:", number_of_labelled_events)
print("Number of labelled channel intervals:", number_of_label_rows)
print("Number of affected channels:", number_of_affected_channels)
print("Number of subsystems involved:", number_of_subsystems)
'''

'\ndata_frame = []\n\nfor zips in unzip_telecommand:\n    with zipfile.ZipFile(zips, "r") as zip:\n        for internal_file in zip.namelist():\n            with zip.open(internal_file) as fileStream:\n                df = pd.read_pickle(fileStream)\n\n                mask = (df.index >= start_time) & (df.index <= end_time)\n                selected_telecommand = df.loc[mask]\n\n                if not selected_telecommand.empty:\n                    telecommand_name = selected_telecommand.columns[0]\n\n                    temp = pd.DataFrame({\n                        "date_time": selected_telecommand.index,\n                        "telecommand": telecommand_name,\n                        "value": selected_telecommand[telecommand_name].values\n                    })\n\n                    data_frame.append(temp)\n\nif data_frame:\n    final_dataFrame = pd.concat(data_frame, ignore_index=True)\n    final_dataFrame = final_dataFrame.sort_values("date_time")\n\n    number_of_telecommand_

### Labels inside the period

In [143]:
'''convert_Stime = pd.to_datetime("2001-11-01 10:30").tz_localize("UTC")
convert_Etime = pd.to_datetime("2001-12-01 16:15").tz_localize("UTC")
labelled_channels["StartTime"]= pd.to_datetime(labelled_channels["StartTime"])
labelled_channels["EndTime"]= pd.to_datetime(labelled_channels["EndTime"])
masked_time = (
    (labelled_channels["StartTime"]>= convert_Stime) &
    (labelled_channels["EndTime"]<= convert_Etime)
)
result = labelled_channels[masked_time]
compare_anamoly = anamoly_types["ID"].isin(result["ID"])
anamoly_result = anamoly_types[compare_anamoly]
detect_anamoly = pd.DataFrame({
    "Event ID": result["ID"],
    "Channel": result["Channel"],
    "Start Time": result["StartTime"],
    "End Time": result["EndTime"],
    "Category": anamoly_result["Category"]
})
print(detect_anamoly)

'''
convert_Stime = pd.Timestamp("2001-11-01 10:30", tz="UTC")
convert_Etime = pd.Timestamp("2001-12-01 16:15", tz="UTC")

labelled_channels["StartTime"] = pd.to_datetime(
    labelled_channels["StartTime"],
    utc=True
)

labelled_channels["EndTime"] = pd.to_datetime(
    labelled_channels["EndTime"],
    utc=True
)

# Select every labelled event that overlaps the chosen period
masked_time = (
    (labelled_channels["StartTime"] <= convert_Etime) &
    (labelled_channels["EndTime"] >= convert_Stime)
)

result = labelled_channels.loc[masked_time].copy()

# Map each event ID to its anomaly category
detect_anamoly = result.merge(
    anamoly_types[["ID", "Category"]],
    on="ID",
    how="left"
)

detect_anamoly = detect_anamoly[
    ["ID", "Channel", "StartTime", "EndTime", "Category"]
].rename(columns={
    "ID": "Event ID",
    "StartTime": "Start Time",
    "EndTime": "End Time"
})

print(detect_anamoly)


  Event ID     Channel                       Start Time  \
0     id_3  channel_14 2001-11-24 17:31:45.561000+00:00   
1     id_3  channel_21 2001-11-24 17:31:45.561000+00:00   
2     id_3  channel_29 2001-11-24 17:31:45.561000+00:00   

                          End Time Category  
0 2001-11-25 00:16:45.561000+00:00  Anomaly  
1 2001-11-24 22:10:45.561000+00:00  Anomaly  
2 2001-11-24 22:36:15.561000+00:00  Anomaly  


#### Affected channels and subsystem metadata

In [146]:
#print(detect_anamoly)

labelled_event_affect = channels.merge(detect_anamoly[["Channel"]], on="Channel", how="right")
print(labelled_event_affect)
    

      Channel    Subsystem    Physical Unit  Group Target
0  channel_14  subsystem_6  physical_unit_3      4    YES
1  channel_21  subsystem_6  physical_unit_3      4    YES
2  channel_29  subsystem_6  physical_unit_3      4    YES


Total telecommand occurrences:
 Unique telecommand types: 
 Most frequent telecommands: 
 Priority distribution:

In [172]:
total_occurance = final_dataFrame["telecommand"].value_counts()
print("Total Occurance", len(total_occurance))

print("Unique Telecommand Types", un_value.nunique())
print("Unique Telecommand Types", un_value.idxmax())

Total Occurance 66
Unique Telecommand Types 28
Unique Telecommand Types telecommand_270
